### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be ensuring the output can be easily parsed and used in subsequent processing. Langchain supports multiple schema types and methods for enforcing structured output

In [ ]:
import os
from langchain.chat_models import init_chat_model
from dotenv import load_dotenv
load_dotenv()

os.environ["GROQ_API_KEY"]
model = init_chat_model("groq:llama-3.3-70b-versatile")
model.profile

#### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [ ]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    title: str = Field(description = "The title of the movie")
    year: int = Field(description = "This year the movie was released")
    director: str = Field(description = "The director of the movie")
    rating: float = Field(description = "The movies rating out of 10")

In [ ]:
structuredModel = model.with_structured_output(Movie)
rawStructuredModel = model.with_structured_output(Movie, include_raw=True)

print("StructuredModel:\n", structuredModel)
print("RawStructuredModel:\n", rawStructuredModel)

In [ ]:
response = structuredModel.invoke("Provide me details about the movie Inception")
rawResponse = rawStructuredModel.invoke("Provide me details about the movie Inception")
print("Response:", response)
print("Raw Response:", rawResponse)

##### Nested Structure

In [ ]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

structuredModel = model.with_structured_output(MovieDetails)

response = structuredModel.invoke("Provide details about the movie Inception")
response

#### Typed Dict

TypedDict provides a simpler alternative using Python's built in typing, ideal when we do not need **runtime validation**

In [ ]:
from typing_extensions import TypedDict, Annotated, NotRequired

class MovieDict(TypedDict):
    '''A movie with details'''
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

In [ ]:
typedDictModel = model.with_structured_output(MovieDict)
response = typedDictModel.invoke("Provide me details about the movie Inception")
response

In [ ]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: NotRequired[float | None]

typedDictModel = model.with_structured_output(MovieDetails)
response = typedDictModel.invoke("Provide me details about the movie Inception")
response

#### Data Classes

A data class is a class typically containing mainly data, although they aren't really any restrictions. You create it using the **@dataclass** decorator

In [ ]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent
from langchain.chat_models import init_chat_model 

class ContactInfo(BaseModel):
    '''Contact information for a person'''
    name: str = Field(description = "The name of the person")
    email: str = Field(description = "The email address of the person")
    phone: str = Field(description = "The phone number of the person")

model = init_chat_model("groq:llama-3.3-70b-versatile")
agent = create_agent(model=model, tools=[], response_format=ContactInfo)

agent

In [ ]:
result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

print(result)
result['structured_response']

In [ ]:
## Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str
    email: str
    phone: str

agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

In [ ]:
## Dataclass

from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str
    email: str
    phone: str


agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]